# 03 · 问答编排与提示词对比

对应任务 **T9**，里程碑 **M3 + M4**。

这一节把「提示词」和「大模型」也接进链路，并用同一个问题并排跑三种风格，
肉眼观察它们的差异。

> 关键：`answer_all_styles` **只检索一次**，三版共用同一份参考资料。
> 这样三版的差异只可能来自提示词风格（§12.1 控制变量）。

In [ ]:
import re

from kbqa.config import settings
from kbqa.embeddings import get_embedding_provider
from kbqa.llm import get_llm
from kbqa.loaders import list_documents
from kbqa.pipeline import RAGPipeline
from kbqa.prompts import PROMPT_STYLES, build_messages
from kbqa.rerank import get_reranker
from kbqa.vectorstore import VectorStore

print("风格：", PROMPT_STYLES)
print(f"召回 top_k = {settings.retrieve_top_k} ｜ 重排 top_n = {settings.rerank_top_n}")

## 1. 准备索引

这次直接用 `pipeline.index_documents()`——它内部做了「先删后加」，所以重复运行是幂等的。

In [ ]:
store = VectorStore(settings.data_dir / "index")
pipeline = RAGPipeline(
    settings=settings,
    store=store,
    embedder=get_embedding_provider(settings),
    reranker=get_reranker(settings),
    llm=get_llm(settings),
)

docs = list_documents(settings.data_dir)
assert docs, "data/uploads 下没有文档，请先放入 .txt/.md（见任务文档 P0-5）"

report = pipeline.index_documents([doc.doc_id for doc in docs])
for row in report:
    print(f"  {row['filename']}  chunk数={row['chunks']}  耗时={row['elapsed_ms']}ms")
print("索引概况：", store.stats())

## 2. 三版并排输出

同一个问题、同一份参考资料，只换 `system` prompt。

观察点（§12.2）：
- `strict_citation` 应当长度最长、分点、几乎每句都带 `[n]`；
- `concise_assistant` 应当只有 1~3 句、一段平实的话；
- `socratic_guide` 应当有「结论 / 依据 / 延伸思考」三段，并以追问收尾。

In [ ]:
QUESTION = "召回和重排有什么区别？各自的取舍是什么？"

results = pipeline.answer_all_styles(QUESTION, PROMPT_STYLES, None)

for result in results:
    print("=" * 72)
    print(f"【{result.style}】  引用数={len(result.citations)}  耗时={result.latency_ms}ms")
    print("=" * 72)
    print(result.answer)
    print()

## 3. 检查引用与结果结构

三个硬性检查：
1. 答案里的每个 `[n]` 都必须落在参考资料范围内（否则就是编造引用）；
2. `candidates` 长度应是 `retrieve_top_k`，`used` 长度应是 `rerank_top_n`；
3. `latency_ms` 应为正数。

In [ ]:
for result in results:
    numbers = [int(m) for m in re.findall(r"\[(\d+)\]", result.answer)]
    assert all(1 <= n <= len(result.used) for n in numbers), (result.style, numbers)
    assert len(result.candidates) == settings.retrieve_top_k, len(result.candidates)
    assert len(result.used) == settings.rerank_top_n, len(result.used)
    assert result.latency_ms > 0, result.latency_ms
    print(f"  {result.style:18s} 引用编号={numbers}  均 <= {len(result.used)} ✓")

assert all(len(r.citations) >= 1 for r in results), "三版都应当给出至少一条引用"
print("✓ 三版都带 [n] 引用，且编号都在参考资料范围内")
print(f"✓ candidates 长度 = {settings.retrieve_top_k}，used 长度 = {settings.rerank_top_n}，latency_ms 均为正数")

## 4. 拒答测试（§12.1 硬约束 2）

问一个**资料里完全没提过**的问题。三版都必须明确拒答，而不是靠模型自己的知识作答。

这里顺便检查一件更重要的事：答案里**不该出现具体数值**——那是编造。

In [ ]:
OFF_TOPIC = "珠穆朗玛峰的海拔是多少米？"
refusals = pipeline.answer_all_styles(OFF_TOPIC, PROMPT_STYLES, None)

for result in refusals:
    print("=" * 72)
    print(f"【{result.style}】  引用数={len(result.citations)}")
    print("=" * 72)
    print(result.answer)
    print()

markers = ("无法回答", "未覆盖", "未提及", "不足以", "没有相关")
for result in refusals:
    assert not re.search(r"88[0-9]{2}", result.answer), f"{result.style} 疑似编造了具体数字"
    assert any(m in result.answer for m in markers), f"{result.style} 没有明确拒答"
print("✓ 三版都明确拒答，且没有编造具体数值")

## 5. 看一眼最终发给 LLM 的 messages（§10 第 8 步）

这是调试提示词工程时最有用的一步：**别猜模型看到了什么，直接打出来。**

In [ ]:
messages = build_messages(PROMPT_STYLES[0], QUESTION, results[0].used)

print("=" * 72)
print(f"[system] 共 {len(messages[0]['content'])} 字")
print("=" * 72)
print(messages[0]["content"])
print()
print("=" * 72)
print(f"[user] 共 {len(messages[1]['content'])} 字")
print("=" * 72)
print(messages[1]["content"])

## 小结

- **三版只差 `system` 的后半段**：硬约束是同一份字符串，风格段各自不同（§12.1 控制变量）。
- **只检索一次**：三版看到完全相同的 4 条参考资料，分差只可能来自提示词。
- **引用是可验证的**：`[n]` → `used[n-1]` → `文件名#chunk序号`，整条链有程序断言兜底。
- **拒答是硬约束**：三版都拒绝回答资料外的问题，且没有编造数值。
- 到这里 **M3 + M4 完成**。下一步 T10 把这些能力包成 HTTP 接口。

> §12.4 的提醒：上面三版是**起点，不是终点**。跑完 T13 的评测后要回来改提示词——
> 「跑 → 看差异 → 改 → 再跑」这轮循环，才是提示词工程真正要练的东西。